# Portfolio Optimizer for Everyday Investors

You have $10,000 in savings and want to invest across a few familiar assets. How do you balance higher expected return with the risk of large losses?

This notebook builds a simple, realistic portfolio optimization workflow using **NVIDIA cuOpt's QP solver** for quadratic programming. We'll start with a small set of assets, compare equal-weight vs. optimized portfolios, and visualize the efficient frontier.

**Key Concepts:**
- Mean-variance optimization (Markowitz portfolio theory)
- Efficient frontier visualization
- Interactive constraint exploration

References:
- cuOpt LP/QP/MILP API Reference: https://docs.nvidia.com/cuopt/user-guide/latest/cuopt-python/lp-qp-milp/lp-qp-milp-api.html
- Portfolio Optimization: https://en.wikipedia.org/wiki/Portfolio_optimization

## Environment Setup and Installation

### Install Required Dependencies

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12 rapids-logger==0.1.19
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13 rapids-logger==0.1.19

In [ ]:
!pip install --extra-index-url https://pypi.nvidia.com "numpy>=1.24.4" "pandas>=2.2.1"

### Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cuopt.linear_programming.problem import (
    Problem,
    QuadraticExpression,
    MINIMIZE,
)

%config InlineBackend.figure_format = "retina"

print("Imports ready (using cuOpt QP solver)")

## **Step 1:** Data Setup - A Small, Realistic Universe

We'll work with a compact set of assets an everyday investor might recognize:
- Cash or money market
- US equity ETF proxy
- International equity ETF proxy
- Bond ETF proxy
- Real-asset/REIT or gold ETF proxy

We'll simulate monthly returns with realistic annual return/volatility assumptions and correlations, then estimate the annualized mean returns and covariance matrix.


In [ ]:
# Simulate monthly returns with realistic assumptions
np.random.seed(7)

assets = ["Cash", "US Equity", "Intl Equity", "Bond", "REIT/Gold"]

annual_mean = np.array([0.02, 0.08, 0.075, 0.04, 0.06])
annual_vol = np.array([0.005, 0.16, 0.18, 0.06, 0.14])

corr = np.array([
    [1.00, 0.05, 0.05, 0.10, 0.05],
    [0.05, 1.00, 0.80, -0.10, 0.55],
    [0.05, 0.80, 1.00, -0.05, 0.50],
    [0.10, -0.10, -0.05, 1.00, 0.00],
    [0.05, 0.55, 0.50, 0.00, 1.00],
])

monthly_mean = annual_mean / 12.0
monthly_vol = annual_vol / np.sqrt(12.0)
monthly_cov = np.outer(monthly_vol, monthly_vol) * corr

n_months = 120
returns = np.random.multivariate_normal(monthly_mean, monthly_cov, size=n_months)

# Estimate annualized mean and covariance from the simulated data
mean_returns = returns.mean(axis=0) * 12.0
cov_matrix = np.cov(returns, rowvar=False) * 12.0

summary = pd.DataFrame(
    {
        "Annualized Return": mean_returns,
        "Annualized Volatility": np.sqrt(np.diag(cov_matrix)),
    },
    index=assets,
)

summary.style.format({"Annualized Return": "{:.2%}", "Annualized Volatility": "{:.2%}"})

## **Step 2:** Baseline Portfolio - Equal-Weight

Before optimizing, let's compute the equal-weight portfolio. This gives a simple baseline to compare against the optimized portfolios.

In [ ]:
def portfolio_stats(weights, mean_vec, cov_mat):
    exp_return = float(weights @ mean_vec)
    variance = float(weights @ cov_mat @ weights)
    volatility = np.sqrt(variance)
    return exp_return, volatility, variance

n_assets = len(assets)
weights_equal = np.repeat(1.0 / n_assets, n_assets)

ret_eq, vol_eq, var_eq = portfolio_stats(weights_equal, mean_returns, cov_matrix)

baseline_df = pd.DataFrame(
    {
        "Weight": weights_equal,
        "Asset": assets,
    }
).set_index("Asset")

baseline_df, ret_eq, vol_eq


## **Step 3:** The Optimization Problem

We translate the investor's goals into a quadratic program (QP):

**Decision variables:** Portfolio weights $w_i$ for each asset $i$.

**Objective:** Minimize portfolio variance (risk):
$$\min \frac{1}{2} w^\top \Sigma w$$
where $\Sigma$ is the covariance matrix.

**Constraints:**
- Fully invested: $\sum_i w_i = 1$
- Long-only (no shorting): $w_i \geq 0$
- Optional: target minimum return $\mu^\top w \geq r_{target}$
- Optional: max allocation per asset $w_i \leq w_{max}$

### cuOpt QP Implementation

cuOpt's QP solver allows us to express quadratic objectives using `Variable * Variable` syntax.
For the portfolio variance $w^\top \Sigma w$, we construct it as:
$$\sum_i \sum_j w_i \cdot \Sigma_{ij} \cdot w_j$$

In [ ]:
def solve_min_variance_qp(
    cov_matrix,
    mean_returns,
    target_return=None,
    max_weight=None,
    min_safe_alloc=None,
    safe_indices=None,
):
    """
    Solve the minimum-variance portfolio problem using cuOpt QP solver.
    
    Parameters
    ----------
    cov_matrix : ndarray (n x n)
        Annualized covariance matrix
    mean_returns : ndarray (n,)
        Annualized expected returns
    target_return : float, optional
        Minimum expected return constraint
    max_weight : float, optional
        Maximum weight for any single asset
    min_safe_alloc : float, optional
        Minimum combined allocation to safe assets
    safe_indices : list of int, optional
        Indices of safe assets (e.g., Cash, Bonds)
    
    Returns
    -------
    weights : ndarray
        Optimal portfolio weights
    portfolio_return : float
        Expected return of the optimal portfolio
    portfolio_vol : float
        Volatility of the optimal portfolio
    status : str
        Solver status
    """
    n = len(mean_returns)
    
    # Create cuOpt Problem
    prob = Problem("Portfolio_Optimization")
    
    # Decision variables: portfolio weights with bounds [0, upper_bound]
    upper_bound = max_weight if max_weight is not None else 1.0
    w = [prob.addVariable(lb=0.0, ub=upper_bound, name=f"w_{i}") for i in range(n)]
    
    # Build quadratic objective: minimize w' * cov_matrix * w
    quad_expr = None
    for i in range(n):
        for j in range(n):
            if abs(cov_matrix[i, j]) > 1e-12:
                if(i==0 and j==0):
                    quad_expr = float(cov_matrix[i, j]) * w[i] * w[j]
                else:
                    quad_expr += float(cov_matrix[i, j]) * w[i] * w[j]
    
    prob.setObjective(quad_expr, sense=MINIMIZE)
    
    # Constraint: Fully invested (sum of weights = 1)
    sum_weights = sum(w)
    prob.addConstraint(sum_weights == 1, name="fully_invested")
    
    # Constraint: Minimum expected return
    if target_return is not None:
        expected_return_expr = sum(mean_returns[i] * w[i] for i in range(n))
        prob.addConstraint(expected_return_expr >= target_return, name="min_return")
    
    # Constraint: Minimum allocation to safe assets
    if min_safe_alloc is not None and safe_indices is not None:
        safe_sum = sum(w[i] for i in safe_indices)
        prob.addConstraint(safe_sum >= min_safe_alloc, name="min_safe")
    
    # Solve the problem
    prob.solve()
    
    # Extract solution
    weights = np.array([w[i].Value for i in range(n)])
    
    # Compute portfolio statistics
    portfolio_return = float(mean_returns @ weights)
    portfolio_vol = np.sqrt(float(weights @ cov_matrix @ weights))
    
    # Get solver status
    status = "optimal" if prob.Status == 1 else f"status_{prob.Status}"
    
    return weights, portfolio_return, portfolio_vol, status

print("Solver function defined (using cuOpt QP)")


## **Step 4:** Minimum-Variance Portfolio (No Return Target)

First, let's find the portfolio with the lowest possible risk, without any constraint on expected return. This is the leftmost point on the efficient frontier.

In [ ]:
weights_mv, ret_mv, vol_mv, status_mv = solve_min_variance_qp(cov_matrix, mean_returns)

print(f"Status: {status_mv}")
print(f"\nMinimum-Variance Portfolio:")
print(f"  Expected Return: {ret_mv:.2%}")
print(f"  Volatility:      {vol_mv:.2%}")
print(f"\nWeights:")
for asset, wt in zip(assets, weights_mv):
    print(f"  {asset:<12}: {wt:6.1%}")

## **Step 5:** Efficient Frontier

The efficient frontier shows all portfolios that offer the highest expected return for each level of risk. We trace it by solving QP problems for a range of target returns.

In [ ]:
# Compute efficient frontier by sweeping target returns
min_ret = ret_mv
max_ret = mean_returns.max()
target_returns = np.linspace(min_ret, max_ret, 20)

frontier_vols = []
frontier_rets = []
frontier_weights = []

for target in target_returns:
    w, r, v, _ = solve_min_variance_qp(cov_matrix, mean_returns, target_return=target)
    frontier_vols.append(v)
    frontier_rets.append(r)
    frontier_weights.append(w)

frontier_vols = np.array(frontier_vols)
frontier_rets = np.array(frontier_rets)

print(f"Computed {len(frontier_rets)} points on the efficient frontier.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Efficient frontier
ax.plot(frontier_vols * 100, frontier_rets * 100, "b-", lw=2, label="Efficient Frontier")

# Individual assets
asset_vols = np.sqrt(np.diag(cov_matrix))
ax.scatter(asset_vols * 100, mean_returns * 100, s=80, c="gray", marker="o", zorder=3, label="Individual Assets")
for i, asset in enumerate(assets):
    ax.annotate(asset, (asset_vols[i] * 100 + 0.3, mean_returns[i] * 100), fontsize=9)

# Equal-weight portfolio
ax.scatter(vol_eq * 100, ret_eq * 100, s=120, c="orange", marker="s", zorder=4, label="Equal-Weight")

# Minimum-variance portfolio
ax.scatter(vol_mv * 100, ret_mv * 100, s=120, c="green", marker="^", zorder=4, label="Min-Variance")

ax.set_xlabel("Volatility (%)")
ax.set_ylabel("Expected Return (%)")
ax.set_title("Efficient Frontier: Risk vs. Return (cuOpt QP)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## **Step 6:** Constrained Portfolio - 5% Target Return + Diversification

Now let's add practical constraints:
- Target at least 5% annual return
- No single asset > 50% (diversification)

In [ ]:
target_5pct = 0.05
max_wt = 0.50

weights_5pct, ret_5pct, vol_5pct, status_5pct = solve_min_variance_qp(
    cov_matrix, mean_returns,
    target_return=target_5pct,
    max_weight=max_wt,
)

print(f"Status: {status_5pct}")
print(f"\nConstrained Portfolio (Target >= 5%, Max Weight 50%):")
print(f"  Expected Return: {ret_5pct:.2%}")
print(f"  Volatility:      {vol_5pct:.2%}")
print(f"\nWeights:")
for asset, wt in zip(assets, weights_5pct):
    print(f"  {asset:<12}: {wt:6.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of weights
x = np.arange(len(assets))
width = 0.35

axes[0].bar(x - width/2, weights_equal * 100, width, label="Equal-Weight", color="orange")
axes[0].bar(x + width/2, weights_5pct * 100, width, label="Optimized (5%+ target)", color="steelblue")
axes[0].set_ylabel("Weight (%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(assets, rotation=15)
axes[0].legend()
axes[0].set_title("Portfolio Allocation Comparison")

# Scatter: return vs vol
axes[1].scatter(vol_eq * 100, ret_eq * 100, s=150, c="orange", marker="s", label=f"Equal-Weight ({ret_eq:.1%}, {vol_eq:.1%})")
axes[1].scatter(vol_5pct * 100, ret_5pct * 100, s=150, c="steelblue", marker="^", label=f"Optimized ({ret_5pct:.1%}, {vol_5pct:.1%})")
axes[1].plot(frontier_vols * 100, frontier_rets * 100, "k--", alpha=0.4, label="Frontier")
axes[1].set_xlabel("Volatility (%)")
axes[1].set_ylabel("Expected Return (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[1].set_title("Return vs. Volatility")

plt.tight_layout()
plt.show()


## **Step 7:** Interactive Exploration - Adjust Your Preferences

Now let's explore how changing your preferences affects the optimal portfolio. You can adjust:
- **Target return**: How much growth do you want?
- **Maximum weight per asset**: Avoid over-concentration
- **Minimum safe allocation**: Keep a floor in Cash + Bonds

In [ ]:
# Interactive exploration function
# In Jupyter, uncomment the interact() call at the bottom to enable sliders

def explore_portfolio(target_return_pct=5.0, max_weight_pct=60.0, min_safe_pct=10.0):
    """Explore portfolio with given parameters."""
    target_return = target_return_pct / 100.0
    max_weight = max_weight_pct / 100.0
    min_safe = min_safe_pct / 100.0
    safe_indices = [0, 3]  # Cash and Bond
    
    w, r, v, status = solve_min_variance_qp(
        cov_matrix, mean_returns,
        target_return=target_return,
        max_weight=max_weight,
        min_safe_alloc=min_safe,
        safe_indices=safe_indices,
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Weights
    colors = ["#2ecc71" if i in safe_indices else "#3498db" for i in range(len(assets))]
    axes[0].barh(assets, w * 100, color=colors)
    axes[0].set_xlabel("Weight (%)")
    axes[0].set_xlim(0, 70)
    axes[0].set_title("Optimal Allocation")
    axes[0].axvline(max_weight * 100, color="red", linestyle="--", label=f"Max {max_weight_pct:.0f}%")
    axes[0].legend()
    
    # Frontier with current point
    axes[1].plot(frontier_vols * 100, frontier_rets * 100, "b-", lw=2, alpha=0.5)
    axes[1].scatter(v * 100, r * 100, s=200, c="red", marker="*", zorder=5, label="Your Portfolio")
    axes[1].scatter(vol_eq * 100, ret_eq * 100, s=80, c="orange", marker="s", zorder=4, label="Equal-Weight")
    axes[1].set_xlabel("Volatility (%)")
    axes[1].set_ylabel("Expected Return (%)")
    axes[1].set_title("Your Portfolio on the Frontier")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Status: {status}")
    print(f"Expected Return: {r:.2%}  |  Volatility: {v:.2%}")
    print(f"Safe Assets (Cash + Bond): {(w[0] + w[3]):.1%}")

# Demo with default parameters (static execution)
explore_portfolio(target_return_pct=5.0, max_weight_pct=60.0, min_safe_pct=10.0)

# To enable interactive sliders in Jupyter, uncomment:
# from ipywidgets import interact, FloatSlider
# interact(
#     explore_portfolio,
#     target_return_pct=FloatSlider(value=5.0, min=2.0, max=8.0, step=0.5, description="Target Return %"),
#     max_weight_pct=FloatSlider(value=60.0, min=25.0, max=100.0, step=5.0, description="Max Weight %"),
#     min_safe_pct=FloatSlider(value=10.0, min=0.0, max=50.0, step=5.0, description="Min Safe %"),
# )

## **Step 8:** Interpretation - What It Means for You

Let's put one scenario in plain English. Suppose you target a 6% annual return with at least 20% in safe assets.

In [ ]:
target_scenario = 0.06
min_safe_scenario = 0.20
safe_indices = [0, 3]

w_scenario, r_scenario, v_scenario, _ = solve_min_variance_qp(
    cov_matrix, mean_returns,
    target_return=target_scenario,
    min_safe_alloc=min_safe_scenario,
    safe_indices=safe_indices,
)

print("=" * 60)
print("YOUR RECOMMENDED PORTFOLIO")
print("=" * 60)
print(f"\nTarget: At least {target_scenario:.0%} annual return")
print(f"Constraint: Keep at least {min_safe_scenario:.0%} in Cash + Bonds\n")

for asset, wt in zip(assets, w_scenario):
    bar = "█" * int(wt * 40)
    print(f"  {asset:<12} {wt:5.1%}  {bar}")

print(f"\n→ Expected Return: {r_scenario:.2%}")
print(f"→ Volatility:      {v_scenario:.2%}")
print(f"→ Safe Allocation: {(w_scenario[0] + w_scenario[3]):.1%}")

print("\n" + "=" * 60)
print("WHAT THIS MEANS")
print("=" * 60)
print(f"""
If you invest $10,000 according to this allocation:
  - Cash:        ${10000 * w_scenario[0]:,.0f}
  - US Equity:   ${10000 * w_scenario[1]:,.0f}
  - Intl Equity: ${10000 * w_scenario[2]:,.0f}
  - Bonds:       ${10000 * w_scenario[3]:,.0f}
  - REIT/Gold:   ${10000 * w_scenario[4]:,.0f}

You can expect ~{r_scenario:.1%} growth per year on average,
with a typical annual swing of about +/-{v_scenario:.1%}.
""")
